## ALE(Accumulated Local Effect) Plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from alibi.explainers.ale import ALE
from alibi.explainers.ale import plot_ale

In [ ]:
# Import dataset
data=pd.read_csv("data/abalone.data",
                 names=['sex', 'length', 'diameter', 'height',
                        'whole weight', 'shucked weight',
                        'viscera weight', 'shell weight', 'rings'])

y=data['rings']

In [ ]:
print(len(data))

In [ ]:
data.head(10)

In [ ]:
x=data[['sex', 'length', 'height', 'shucked weight',
        'viscera weight', 'shell weight']]

In [ ]:
# Create dummy variables
x['sex.M']=[1 if s=='M' else 0 for s in x['sex']]
x['sex.F']=[1 if s=='F' else 0 for s in x['sex']]
x['sex.I']=[1 if s=='I' else 0 for s in x['sex']]

x=x.drop('sex', axis=1)

x.head(10)

In [ ]:
# Train the model
rf=RandomForestRegressor()
rf.fit(x.to_numpy(), y)

In [ ]:
# Get predictions
y_pred=rf.predict(x)

# Model evaluation
fig, ax=plt.subplots(nrows=1, ncols=1, figsize=(8, 8))

plt.scatter(y, y_pred)
plt.plot([y.min(), y.max()], [y.min(), y.max()], color='tab:red')

plt.ylabel('Predicted', size=15)
plt.xlabel('Actual', size=15)

### ALEs

In [ ]:
# Get ALE explanation
ale=ALE(
    predictor=rf.predict,
    feature_names=x.columns,
    target_names=['rings']
)

exp=ale.explain(x.to_numpy())

In [ ]:
# Plot ALE explanation for first 3 features
plot_ale(exp=exp, features=[0, 1, 2], fig_kw={'figwidth': 15, 'figheight': 5})

- The effect of `length` and `height` on the predicted number of rings is lower when compared to shucked weight.

- The `downward sloping line` for shucked weight indicates that the predicted number of rings tends to decrease as the shucked weight increases.

In [ ]:
# Plot ALE for weight features
fig, ax=plt.subplots(1, 1, figsize=(8, 4))

plot_ale(exp, features=[2], ax=ax, line_kw={'label': 'shucked weight'})
plot_ale(exp, features=[3], ax=ax, line_kw={'label': 'viscera weight'})
plot_ale(exp, features=[4], ax=ax, line_kw={'label': 'shell weight'})

ax.set_xlabel('weight')

In [ ]:
# Adjust the intervals
ale=ALE(
    predictor=rf.predict,
    feature_names=x.columns,
    target_names=['rings']
)
exp=ale.explain(x.to_numpy(), min_bin_points=50)

fig, ax=plt.subplots(1, 1, figsize=(8, 4))

plot_ale(exp, features=[2], ax=ax, line_kw={'label': 'shucked weight'})
plot_ale(exp, features=[3], ax=ax, line_kw={'label': 'viscera weight'})
plot_ale(exp, features=[4], ax=ax, line_kw={'label': 'shell weight'})

ax.set_xlabel('weight')